# Demo 4jun25

## Programmatic methods to sample & compare NS and SRNS points

In [191]:
# Custom modules with their classes inside
import behaviors  # noqa: I001
import no_signaling_sets

import numpy as np

import sys
sys.path.append('../code_legacy')
import  extract_hyperplanes # Unfinished module# type: ignore # noqa: I001


## Experiment parameters

In [192]:
delta = 2 # Number of outputs a,b
m = 2     # Number of inputs x,y

## Matrix form of latent srns

In [193]:
latent_set = no_signaling_sets.LatentSRNSSet(delta, m)

A_latent, b_latent = latent_set.get_equations()
print("A_latent:")
print(A_latent.shape)
print("b_latent:")
print(b_latent.shape)

A_latent:
(26, 32)
b_latent:
(26,)


## Get a non-SRNS point from database

In [194]:
data = np.load("../data/non_srns/non_srns_points.npy")
example_point = data[np.random.randint(len(data))]

print("Example point as vector:", example_point)
print("\n\n")
print("Example point as behavior:")
behavior = behaviors.RoutedBehavior(delta, m, example_point)
print(behavior)

Example point as vector: [0.32164616 0.27299275 0.13726344 0.05421783 0.04089039 0.0895438
 0.15024784 0.23329346 0.14270046 0.34796776 0.32708318 0.56674268
 0.49476299 0.28949569 0.38540554 0.14574603 0.20319766 0.0156959
 0.27200155 0.21729547 0.15933889 0.34684065 0.01550973 0.07021581
 0.13375861 0.26818219 0.06495472 0.06658262 0.50370484 0.36928126
 0.647534   0.6459061 ]



Example point as behavior:
Behavior:
Short path (z=S):
[[0.32164616 0.27299275 0.13726344 0.05421783]
 [0.04089039 0.0895438  0.15024784 0.23329346]
 [0.14270046 0.34796776 0.32708318 0.56674268]
 [0.49476299 0.28949569 0.38540554 0.14574603]]
Long path (z=L) :
[[0.20319766 0.0156959  0.27200155 0.21729547]
 [0.15933889 0.34684065 0.01550973 0.07021581]
 [0.13375861 0.26818219 0.06495472 0.06658262]
 [0.50370484 0.36928126 0.647534   0.6459061 ]]
------------


### Elementary tests on behaviors

In [195]:
print(f"Coordinates are positive                      : {behavior.positivity()}")
print(f"Coordinates are normalized                    : {behavior.normalization()}")
print(f"Coordinates verify the no-signaling conditions: {behavior.no_signaling()}")
print()
print("Aggregated tests (checks all previous tests):")
print(f"  Normalization                               : {behavior.is_normalized()}")
print(f"  No-signaling                                : {behavior.is_no_signaling()}")

Coordinates are positive                      : True
Coordinates are normalized                    : True
Coordinates verify the no-signaling conditions: True

Aggregated tests (checks all previous tests):
  Normalization                               : True
  No-signaling                                : True


## Instanciate a set to test for belonging in SRNS

In [196]:
srns_set = no_signaling_sets.ShortRangeNoSignalingSet(delta, m)
srns_set

### Pipelined belonging test

In [197]:
print(f"Does SRNS set contains the example point: {srns_set.is_in_set(behavior)}")

Does SRNS set contains the example point: False


### Main steps to belonging test

#### [1] Get the equation to test for belonging in matrix form

In [198]:
A,b = srns_set.get_equations(behavior)

with np.printoptions(threshold=np.inf, linewidth=np.inf, precision=2): # type: ignore
    print("A matrix:")
    print(A[:-4])
    print("Last 4 rows of A matrix")
    print(A[-4:])
    print()
    print("b vector:")
    print(b)

A matrix:
[[-0.07  1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [-0.02  0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.11  0.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.2   0.    0.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.21  0.    0.    0.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0. 

#### [2] Solve the linear program

In [199]:
from scipy.optimize import OptimizeResult

result: OptimizeResult = srns_set.lp_test(behavior)

In [200]:
print(f"Alpha value for  alpha*p + (1-alpha)*I  : {-result.fun}", "< 1" if -result.fun < 1 else ">= 1")  # noqa: E501

print()

latent_q_vector = result.x[1:]
latent_behavior = behaviors.LatentSRNSBehavior(delta, m, latent_q_vector)
print("Latent behavior:")
print(latent_behavior)
print("Latent behavior is no-signaling: ", latent_behavior.no_signaling())

Alpha value for  alpha*p + (1-alpha)*I  : 0.9052539047057228 < 1

Latent behavior:
Behavior:
Short path (z=S):
[[0.31485796 0.27081428 0.14794479 0.07276742]
 [0.06070271 0.1047464  0.15969897 0.23487634]
 [0.15286667 0.3386857  0.31977985 0.53673255]
 [0.47157265 0.28575363 0.37257639 0.15562369]]
Long path (z=L) :
[[0.0378953  0.18266733]
 [0.1697367  0.08724966]
 [0.         0.03772676]
 [0.16792868 0.        ]
 [0.14477203 0.        ]
 [0.         0.08248704]
 [0.12168746 0.0839607 ]
 [0.35797983 0.52590851]]
------------
Latent behavior is no-signaling:  True


In [201]:
print(
    "Latent matrix tests correctly for SRNS belonging: ",
    latent_behavior.no_signaling() and np.allclose(
        A_latent @ latent_behavior.get_vector(),
        b_latent,
        atol=1e-10,
        )
)


Latent matrix tests correctly for SRNS belonging:  True


In [202]:
def format_lambda_to_hyperplane(lam: np.ndarray) -> str:
    return str(lam[:16]) + str(lam[16:32]) + str(lam[32:]) 

In [203]:
_, _, lambda_var = srns_set.is_facet_hyperplane(behavior)
hyperplanes_extractor = extract_hyperplanes.HyperplanesExtractor(delta, m, list(data))
print("Corresponding lambda variable:")
with np.printoptions(threshold=np.inf, precision=2):  # type: ignore
    print(lambda_var)
print(len(lambda_var), "is the number of coordinates in the dual variable")
print()

hyperplane = hyperplanes_extractor.scale_down_vector(lambda_var)
print("Rescaled and sliced to the hyperplane size:")
print(format_lambda_to_hyperplane(hyperplane))

Corresponding lambda variable:
[ 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    1.81  0.
  0.    0.    1.81  0.   -1.81  1.81  1.81 -1.81  0.    0.    1.81  0.  ]
36 is the number of coordinates in the dual variable

Rescaled and sliced to the hyperplane size:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0][ 0  0  0  0  0  0  1  0  0  0  1  0 -1  1  1 -1][0 0 1 0]


# Sampling

In [204]:
import samplers

In [205]:
sampler = samplers.NoSignalingSampler(delta, m, True)

In [206]:
sampled_vec = sampler.sample()
print("Sampled vector:")
print(sampled_vec)

2025-06-06 11:50:30.462 | SUCCESS  | samplers:sample_multiple:116 - Samples shape: (1, 32)


Sampled vector:
Behavior:
Short path (z=S):
[[0.22053695 0.34928825 0.23501682 0.52091474]
 [0.34627807 0.21752677 0.4798675  0.19396959]
 [0.04948531 0.24926827 0.03500543 0.07764178]
 [0.38369967 0.18391671 0.25011024 0.20747389]]
Long path (z=L) :
[[0.13089734 0.1868842  0.38302397 0.2751587 ]
 [0.43591767 0.37993082 0.33186036 0.43972563]
 [0.31120821 0.26586883 0.05908158 0.17759433]
 [0.12197677 0.16731615 0.22603409 0.10752134]]
------------


In [207]:
srns_set

In [208]:
identity = behaviors.completely_mixed_behavior


In [209]:
result = srns_set.lp_test(identity)

In [210]:
-result.fun

2.0

# Post-demo : testing on quantum NS distributions

In [211]:
from qutip import Qobj, basis, expect, ket2dm, qeye, sigmax, sigmaz, tensor

# AI GENERATED CODE

def projectors(op: Qobj):
    """Return projectors Π_{+1} and Π_{-1} for a Hermitian observable with eigenvalues ±1"""
    eigvals, eigvecs = op.eigenstates()
    proj_dict = {}
    for val, vec in zip(eigvals, eigvecs):
        key = int(np.sign(val)) if val != 0 else +1  # Disambiguate 0 as +1
        proj_dict[key] = ket2dm(vec)
    return proj_dict[+1], proj_dict[-1]

def bell_conditional_distribution(
    alice_ops: list[Qobj],
    bob_ops: list[Qobj]
) -> dict[tuple[int, int], dict[tuple[int, int], float]]:
    """Compute P(a, b | x, y) for all a,b ∈ {±1}, x in Alice ops, y in Bob ops"""

    # Create Φ⁺ state
    ket0 = basis(2, 0)
    ket1 = basis(2, 1)
    phi_plus = (tensor(ket0, ket0) + tensor(ket1, ket1)).unit()
    rho = ket2dm(phi_plus)

    outcomes = [-1, +1]
    distribution = dict()

    for x_index, Ax in enumerate(alice_ops):
        for y_index, By in enumerate(bob_ops):
            # Projectors Π^A_a ⊗ Π^B_b
            Pa_pos, Pa_neg = projectors(Ax)
            Pb_pos, Pb_neg = projectors(By)

            Pxy = dict()
            for a in outcomes:
                Pa = Pa_pos if a == +1 else Pa_neg
                for b in outcomes:
                    Pb = Pb_pos if b == +1 else Pb_neg
                    M = tensor(Pa, Pb)
                    prob = (M * rho).tr().real
                    Pxy[(a, b)] = round(prob, 10)  # Clean rounding
            distribution[(x_index, y_index)] = Pxy

    return distribution
